# 01 - Dataset inspection (EMSCAD)

Thin notebook over `backend/services/dataset.py`: all logic lives in tested code, so the notebook and the CLI cannot disagree.

**Status: not executed in the authoring environment** (the real file was unreachable there). Run all cells after placing `data/raw/fake_job_postings.csv`.

Test-set rule: this notebook never loads `test.csv`.

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd
from backend.services import dataset as ds, dq_report
pd.set_option('display.max_colwidth', 120)
RAW = ROOT / 'data/raw/fake_job_postings.csv'
print(RAW, RAW.exists())

## 1. Load and validate schema

In [ ]:
raw = ds.load_raw(RAW)
print(ds.validate_schema(raw))
raw.dtypes

## 2. Labels and missing values

In [ ]:
df = ds.add_model_text(raw)
prof = ds.profile(df)
print(prof['label_counts'], prof['fraud_rate'])
pd.DataFrame(prof['missingness']).T

## 3. Text length and truncation risk

In [ ]:
pd.DataFrame(prof['text_length_words']).T

## 4. Contact fields: redacted or not?
Look at the counts, then read a few rows yourself.

In [ ]:
print(json.dumps(prof['contact_patterns'], indent=2))
raw[raw[ds.TEXT_FIELDS].fillna('').astype(str).agg(' '.join, axis=1).str.contains(ds._PLACEHOLDER)].head(5)[ds.TEXT_FIELDS]

## 5. Duplicates
Inspect groups before trusting the threshold.

In [ ]:
group, dup = ds.assign_duplicate_groups(df)
df[ds.GROUP_COL] = group
print(json.dumps({k: v for k, v in dup.items() if k != 'group_size_distribution'}, indent=2))
big = df.groupby(ds.GROUP_COL).filter(lambda g: len(g) > 1).sort_values([ds.GROUP_COL, ds.ID_COL])
big[[ds.GROUP_COL, ds.ID_COL, ds.LABEL_COL, 'title']].head(40)

## 6. Gates (mechanical; licence is manual)

In [ ]:
res = {'profile': prof, 'duplicates': dup}
for g, s, d in dq_report.gates(res): print(f'[{s:6}] {g}: {d}')

## 7. Create the shared split
Run `python scripts/prepare_data.py` from the repo root (it writes `data/processed/*.csv` and `split_manifest.json` and the report). Then check the manifest here.

In [ ]:
manifest = json.loads((ROOT / 'data/processed/split_manifest.json').read_text())
print(json.dumps({k: manifest[k] for k in ('seed', 'fractions', 'splits', 'leakage')}, indent=2))
train = ds.load_split('train', ROOT / 'data/processed')
val = ds.load_split('val', ROOT / 'data/processed')
print(len(train), len(val))